In [ ]:
import behaviors
import no_signaling_sets
import numpy as np
from tqdm import tqdm

## Parameters

In [3]:
delta=2
m=2

n_hyperplanes = int(1e5)

## Hyperplane Extraction and Analysis

### One sample plane

In [4]:
non_srns_samples = np.load("../data/non_srns/non_srns_points.npy")
# non_srns_samples = list(non_srns_samples)

example_sample = np.random.choice(len(non_srns_samples), 1)[0]
example_sample = non_srns_samples[example_sample]
example_sample = behaviors.RoutedBehavior(delta, m, vector=example_sample)

srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

print(f"Example sample: {example_sample}")
print(f"Example sample is no-signaling: [{example_sample.is_no_signaling()}]")

np.linalg.matrix_rank(np.array(non_srns_samples))

Example sample: Behavior:
Short path (z=S):
[[0.00698888 0.2283022  0.23212103 0.31108967]
 [0.34696664 0.12565332 0.26813488 0.18916624]
 [0.56117443 0.26535475 0.33604228 0.18256729]
 [0.08487005 0.38068973 0.16370181 0.31717681]]
Long path (z=L) :
[[0.28038947 0.09202855 0.08023185 0.08647623]
 [0.07356605 0.26192697 0.42002406 0.41377967]
 [0.18657242 0.4828711  0.38673005 0.48842342]
 [0.45947206 0.16317338 0.11301405 0.01132068]]
------------
Example sample is no-signaling: [True]


np.int64(15)

### Some rank estimation & stuff

In [5]:
family = non_srns_samples[:n_hyperplanes]
family_rank = np.linalg.matrix_rank(family)
print(f"Rank of family: {family_rank}")

chosen_hyperplane = srns_set.get_facet_hyperplane(example_sample)
hyperplane_behavior = behaviors.RoutedBehavior(delta, m, vector=chosen_hyperplane)

print(f"Hyperplane: {chosen_hyperplane}")
print(f"In behavior expression: {hyperplane_behavior}")

2025-05-26 18:14:17.767 | WARNING  | no_signaling_sets:get_facet_hyperplane:384 - The has rank 11, not yet identified as maximal hyperplane dimension.


Rank of family: 15
Hyperplane: [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  1.67557964  0.          0.          0.          1.67557964 -1.67557964
  0.          0.          0.          0.         -1.67557964  1.67557964
  1.67557964  0.        ]
In behavior expression: Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[ 0.          0.          1.67557964  0.        ]
 [ 0.          0.          1.67557964 -1.67557964]
 [ 0.          0.          0.          0.        ]
 [-1.67557964  1.67557964  1.67557964  0.        ]]
------------


## Bulk process samples for all hyperplanes

In [ ]:
def scale_down_vector(vector: np.ndarray) -> np.ndarray:
    mask = np.abs(vector) > 1e-10

    factor = np.abs(vector[mask])[0]

    abs_is_cst = np.allclose(np.abs(vector[mask]), factor, atol=1e-10)
    # print(f"{np.abs(vector[mask])} == {factor} ? {abs_is_cst}")
    if abs_is_cst:
        # Divide by the factor and round to nearest integer to avoid floating point issues
        rescaled = np.zeros_like(vector)
        rescaled[mask] = np.round(vector[mask] / factor).astype(int)
        return rescaled
    else:
        raise ValueError(f"Vector cannot be scaled down uniformly : {vector}")

def normalize_vector(vector: np.ndarray) -> np.ndarray:
    if np.allclose(vector, 0, atol=1e-10):
        return vector
    else:
        return vector / np.linalg.norm(vector)


In [ ]:
all_hyperplanes_scaled = set()
all_hyperplanes_normalized = set()

i = 0
for vec in list(non_srns_samples)[:n_hyperplanes]:
    _, _, vec_lambda = srns_set.is_facet_hyperplane(
        behaviors.RoutedBehavior(delta, m, vec)
        )
    rescaled_vec = scale_down_vector(vec_lambda)

    if str(rescaled_vec) not in all_hyperplanes_scaled:
        all_hyperplanes_scaled.add(str(rescaled_vec))
        i += 1
        print(f"[{i}] {behaviors.RoutedBehavior(delta, m, rescaled_vec[:32])}")

for vec in tqdm(list(non_srns_samples)[:n_hyperplanes]):
    _, _, vec_lambda = srns_set.is_facet_hyperplane(
        behaviors.RoutedBehavior(delta, m, vec)
        )
    normalized_vec = normalize_vector(vec_lambda)

    all_hyperplanes_normalized.add(str(normalized_vec))

In [8]:
# import re

# print(f"Number of hyperplanes scaled: {len(all_hyperplanes_scaled)}")
# copy_hyperplanes = list(all_hyperplanes_scaled.copy())
# copy_hyperplanes.sort()
# for hyperplane in copy_hyperplanes:
#     printable = hyperplane.replace("\n", "").replace(".", "")

#     regex_1 = re.sub(r"(\d)\s(\d)", r'\1  \2', printable)
#     regex_2 = re.sub(r"(\d)\s(\d)", r'\1  \2', regex_1)
#     final = re.sub(r"(\[)(\d)", r'\1 \2', regex_2)

#     print(final)

# print(f"Number of hyperplanes normalized: {len(all_hyperplanes_normalized)}")
# copy_hyperplanes_normalized = list(all_hyperplanes_normalized.copy())
# copy_hyperplanes_normalized.sort()
# for hyperplane in copy_hyperplanes_normalized:
#     printable = hyperplane.replace("\n", "")

#     regex_1 = re.sub(r"0\.(\d{7,11})", r'1.', printable)
#     regex_2 = re.sub(r"(\d\.)\s\s+(\d\.)", r'\1 \2', regex_1)
#     regex_2 = re.sub(r"(\d\.)\s\s+(\d\.)", r'\1 \2', regex_2)
#     regex_3 = re.sub(r"(\d)\.\s(\d\.)", r'\1  \2', regex_2)
#     regex_3 = re.sub(r"(\d)\.\s(\d)", r'\1  \2', regex_3)
#     regex_4 = re.sub(r"(\d)\.\s+(\-\d)", r'\1 \2', regex_3)
#     final = re.sub(r"(\d)\s*(\])", r'\1\2', regex_4.replace(".", ""))

#     print(final)

In [9]:
A, b = srns_set.get_equations(example_sample)

with np.printoptions(precision=1, threshold=np.inf):
    for line in A[-4:,1:]:
        print(str(line).replace(".", "").replace("\n", ""))


[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1]


## With (2,2,2) hyperplanes in a file, do more work on it

In [10]:
with open("../data/extracted_srns_hyperplanes_comparison.txt", "r") as f:
    lines = f.readlines()

scaled_equations = [line.strip("[]. \n").replace("  ", " ").split(" ") for line in lines[1:33]]
normalized_equations = [line.strip("[]. \n").replace("  "," ").split(" ") for line in lines[34:66]]

print(f"Scaled equations: {len(scaled_equations)}, size {len(scaled_equations[0])}")
print(f"Normalized equations: {len(normalized_equations)}, size {len(normalized_equations[0])}")

scaled_equations = np.array(scaled_equations, dtype=int)
normalized_equations = np.array(normalized_equations, dtype=int)

# scaled_equations = sorted([tuple(row) for row in scaled_equations])
# normalized_equations = sorted([tuple(row) for row in normalized_equations])

# scaled_equations = np.array(scaled_equations)
# normalized_equations = np.array(normalized_equations)

Scaled equations: 32, size 36
Normalized equations: 32, size 36


### Trying to find equivalence classes for each equation

In [11]:
# class HashableArray:
#     def __init__(self, arr):
#         self.array = np.array(arr, dtype=int)

#     def __hash__(self):
#         return hash(self.array.tobytes())

#     def __eq__(self, other):
#         if isinstance(other, HashableArray):
#             return np.array_equal(self.array, other.array)
#         return False

#     def __str__(self):
#         return str(self.array.__str__())

#     def __repr__(self):
#         return self.__str__()

def format_hyperplane(equation: np.ndarray) -> tuple[np.ndarray]:
    # By convention, we give equations a sign such that the first non-zero element is positive.
    first_nonzero_idx = np.flatnonzero(equation)[0]
    if equation[first_nonzero_idx] < 0:
        equation = -equation

    el_S, el_L, el_NS = equation[0:delta**2 * m**2], equation[delta**2 * m**2:2 * delta**2 * m**2], equation[2 * delta**2 * m**2:]  # noqa: E501
    el_S = el_S.reshape((delta, delta, m, m))
    el_L = el_L.reshape((delta, delta, m, m))
    el_NS = el_NS.reshape((delta,) * m)
    return (el_S, el_L, el_NS)

def format_list_of_hyperplanes(equations: list[list[np.ndarray]]) -> list[tuple[np.ndarray]]:
    formatted: list[tuple[np.ndarray]] = []
    for el in equations:
        formatted.append(format_hyperplane(el))

    return formatted

def flatten_hyperplane(hyperplane: tuple[np.ndarray, np.ndarray, np.ndarray]) -> np.ndarray:
    el_S, el_L, el_NS = hyperplane

    equation = np.concatenate((el_S.flatten(), el_L.flatten(), el_NS.flatten()))

    # By convention, we give equations a sign such that the first non-zero element is positive.
    first_nonzero_idx = np.flatnonzero(equation)[0]
    if equation[first_nonzero_idx] < 0:
        equation = -equation

    return equation

def flatten_list_of_hyperplanes(
        hyperplanes: list[tuple[np.ndarray, np.ndarray, np.ndarray]],
        ) -> np.ndarray:
    flattened = []
    for hyperplane in hyperplanes:
        flattened.append(flatten_hyperplane(hyperplane))
    return np.array(flattened, dtype=int)

In [12]:
from itertools import permutations


def permute_a(formatted_hyperplane:tuple[np.ndarray]) -> list[tuple[np.ndarray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of a values."""
    permuted_hyperplanes = []
    for perm in permutations(range(delta)):
        permuted_S = formatted_hyperplane[0][list(perm), :, :, :]
        permuted_L = formatted_hyperplane[1][list(perm), :, :, :]
        # permuted_S, permuted_L = HashableArray(permuted_S), HashableArray(permuted_L)
        permuted_NS = formatted_hyperplane[2]

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

def permute_b(formatted_hyperplane:tuple[np.ndarray]) -> list[tuple[np.ndarray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of b values."""
    permuted_hyperplanes = []
    for perm in permutations(range(delta)):
        permuted_S = formatted_hyperplane[0][:, list(perm), :, :]
        permuted_L = formatted_hyperplane[1][:, list(perm), :, :]
        permuted_NS = formatted_hyperplane[2][np.ix_(*([list(perm)] * m))]
        # permuted_S, permuted_L, permuted_NS = HashableArray(permuted_S), HashableArray(permuted_L), HashableArray(permuted_NS)  # noqa: E501

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

def permute_x(formatted_hyperplane:tuple[np.ndarray]) -> list[tuple[np.ndarray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of x values."""
    permuted_hyperplanes = []
    for perm in permutations(range(m)):
        permuted_S = formatted_hyperplane[0][:, :, list(perm), :]
        permuted_L = formatted_hyperplane[1][:, :, list(perm), :]
        # permuted_S, permuted_L = HashableArray(permuted_S), HashableArray(permuted_L)
        permuted_NS = formatted_hyperplane[2]

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

def permute_y(formatted_hyperplane:tuple[np.ndarray]) -> list[tuple[np.ndarray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of y values."""
    permuted_hyperplanes = []
    for perm in permutations(range(m)):
        permuted_S = formatted_hyperplane[0][:, :, :, list(perm)]
        permuted_L = formatted_hyperplane[1][:, :, :, list(perm)]
        # permuted_S, permuted_L = HashableArray(permuted_S), HashableArray(permuted_L)
        permuted_NS = formatted_hyperplane[2]

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

list_of_permutations = [permute_a, permute_b, permute_x, permute_y]

def permute(axis: int|str, hyperplane: np.ndarray) -> list[np.ndarray]:
    """Permute a hyperplane along the specified axis."""
    formatted_hyperplane = format_hyperplane(hyperplane)

    if isinstance(axis, str):
        axis = axis.lower()
        if axis == "a":
            unformatted_res = permute_a(formatted_hyperplane)
        elif axis == "b":
            unformatted_res = permute_b(formatted_hyperplane)
        elif axis == "x":
            unformatted_res = permute_x(formatted_hyperplane)
        elif axis == "y":
            unformatted_res = permute_y(formatted_hyperplane)
        else:
            raise ValueError(f"Unknown axis: {axis}")
    elif isinstance(axis, int):
        if 0 <= axis < 4:
            unformatted_res = list_of_permutations[axis](formatted_hyperplane)
        else:
            raise ValueError(f"Axis index out of range: {axis}")
    else:
        raise TypeError(f"Axis must be an int or a str, got {type(axis)}")

    return flatten_list_of_hyperplanes(unformatted_res)


In [13]:
# def quotient_set(equations: list[np.ndarray]) -> list[list[np.ndarray]]:
#     """Quotient the set of hyperplanes by the action of the permutation group."""
#     quotient = []

#     for eq in equations:
#         is_represented = False
#         for rep in quotient:
#             if any(
#                 np.array_equal(eq, rep_elem) or np.array_equal(eq, -rep_elem)
#                 for rep_elem in rep
#                 ):
#                 # print(f"HIT !\n{eq} is represented by\n{rep}")
#                 is_represented = True
#                 break

#         if not is_represented:
#             # We need to represent all permuations of the hyperplane equation,
#             # to accurately capture the equivalence class.
#             # There are (delta!)^2 * (m!)^2 permutations of the hyperplane.
#             equations_set = [eq]

#             for axis in range(4):
#                 new_equations_set = []
#                 for e in equations_set:
#                     permuted = permute(axis, e)
#                     for p in permuted:
#                         # Use array_equal for tuple of arrays comparison
#                         if not any(
#                             np.array_equal(p, q) or np.array_equal(p, -q)
#                             for q in new_equations_set
#                             ):
#                             new_equations_set.append(p)
#                 equations_set = new_equations_set

#             quotient.append(equations_set)

#     return quotient

In [14]:
def canonicalize(eq: np.ndarray) -> np.ndarray:
    # Flatten then force the first nonzero element to be positive
    flat = eq.flatten()
    idx = np.flatnonzero(flat)
    if idx.size and flat[idx[-1]] < 0:
        flat = -flat
    return flat

def full_orbit(eq: np.ndarray) -> list[np.ndarray]:
    # Build the full orbit under the permutation group iteratively.
    canon = canonicalize(eq)
    orbit = {canon.tobytes()}  # using bytes as a key
    frontier = [canon]

    while frontier:
        new_frontier = []
        for element in frontier:
            for axis in range(4):
                # permute returns a list of flattened, sign-normalized equations.
                for permuted in permute(axis, element):
                    perm_canon = canonicalize(permuted)
                    key = perm_canon.tobytes()
                    if key not in orbit:
                        orbit.add(key)
                        new_frontier.append(perm_canon)
        frontier = new_frontier
    # Convert each stored bytes back to np.ndarray.
    return [np.frombuffer(key, dtype=element.dtype).reshape(element.shape) for key in orbit]

# def quotient_set(equations: list[np.ndarray]) -> list[list[np.ndarray]]:
#     """Group equations into equivalence classes, where two equations that are
#     the same up to a sign and permutation are equivalent."""
#     quotient = []

#     for eq in equations:
#         canon = canonicalize(eq)
#         # Check if a representative of its orbit was already seen.
#         found = False
#         for rep in quotient:
#             # Assume rep[0] is the canonical representative.
#             if np.array_equal(canonicalize(rep[0]), canonicalize(canon)):
#                 found = True
#                 break
#         if not found:
#             orbit = full_orbit(eq)
#             quotient.append(orbit)
#     return quotient

def quotient_set(equations: list[np.ndarray]) -> list[list[np.ndarray]]:
    """Group equations into equivalence classes (merging overlapping orbits),
    where two equations that are the same up to a sign and permutation are equivalent."""
    # Use sets of canonical bytes to represent an orbit
    orbits: list[set[bytes]] = []  # list of sets of bytes
    for eq in equations:
        # Build the orbit for this equation and convert to bytes (canonicalized)
        orbit_arr = full_orbit(eq)
        orbit_bytes = { canonicalize(x).tobytes() for x in orbit_arr }
        merged = False
        for existing in orbits:
            # If they share any element, merge the orbits
            if not orbit_bytes.isdisjoint(existing):
                existing.update(orbit_bytes)
                merged = True
                break
        if not merged:
            orbits.append(orbit_bytes)

    # Convert each orbit (set of bytes) back to numpy arrays.
    # We assume all equations share the same shape and dtype, so we use the first equation's.
    result = []
    for orbit in orbits:
        group = []
        for byte_key in orbit:
            arr = np.frombuffer(byte_key, dtype=equations[0].dtype).reshape(equations[0].shape)
            group.append(arr)
        result.append(group)
    return result



In [21]:
scaled_quotient = quotient_set(scaled_equations)
normal_quotient = quotient_set(normalized_equations)

def quotient_sets_equal(
        q1: list[list[np.ndarray]],
        q2: list[list[np.ndarray]],
        ) -> bool:
    if len(q1) != len(q2):
        return False

    # Enforce a canonical representation for comparison
    comp_q1 = [[canonicalize(eq) for eq in class1] for class1 in q1]
    comp_q2 = [[canonicalize(eq) for eq in class2] for class2 in q2]

    # Convert to bytes for consistent ordering
    comp_q1 = [[eq.tobytes() for eq in class1] for class1 in q1]
    comp_q2 = [[eq.tobytes() for eq in class2] for class2 in q2]

    # Sort the classes to ensure order does not matter
    for i in range(len(comp_q1)):
        comp_q1[i].sort()
        comp_q2[i].sort()

    comp_q1.sort()
    comp_q2.sort()

    for class1, class2 in zip(comp_q1, comp_q2):
        if len(class1) != len(class2):
            print(f"Class size mismatch: {len(class1)} != {len(class2)}")
            return False

        # Each class is a list of tuples of arrays
        for t1, t2 in zip(class1, class2):
            if not np.array_equal(np.frombuffer(t1, dtype=int), np.frombuffer(t2, dtype=int)):
                print(f"Mismatch in class elements: {t1} != {t2}")
                return False

    return True

print(f"Scaled quotient set: {len(scaled_quotient)} equivalence classes")
# print(f"Scaled quotient set: {scaled_quotient}")
print()
print(f"Normal quotient set: {len(normal_quotient)} equivalence classes")
# print(f"Normal quotient set: {normal_quotient}")
print()
print(f"Comparison: {quotient_sets_equal(scaled_quotient, normal_quotient)}")

all_equations1 = set()
all_equations2 = set()
for class1 in scaled_quotient:
    all_equations1.update({ canonicalize(eq).tobytes() for eq in class1 })
for class2 in normal_quotient:
    all_equations2.update({ canonicalize(eq).tobytes() for eq in class2 })

print(f"Number of unique equations in scaled quotient: {len(all_equations1)}")
print(f"Number of unique equations in normal quotient: {len(all_equations2)}")
print(f"Are the sets of unique equations equal? {all_equations1 == all_equations2}")
print(f"Equivalence classes computed sizes is {len(all_equations1) / 16}")

Scaled quotient set: 16 equivalence classes

Normal quotient set: 16 equivalence classes

Comparison: True
Number of unique equations in scaled quotient: 256
Number of unique equations in normal quotient: 256
Are the sets of unique equations equal? True
Equivalence classes computed sizes is 16.0


In [31]:
def find_og(eq: np.ndarray, equations: list[np.ndarray]) -> np.ndarray:
    """Find the original equation in the list of equations."""
    for original in equations:
        if np.array_equal(canonicalize(original), canonicalize(eq)):
            return original
    return None

print("Representatives of equivalence classes")
print("""
      Canonicalization has been applied,
      and thus we do no know the actual sign of the normal vector
      when the original equation cannot be found.
      It may point inwards or outwards.
      """)
for i, class1 in enumerate(scaled_quotient):
    print(f"Class {i+1}:")
    for eq in class1:
        original_eq = find_og(eq, scaled_equations)
        if original_eq is not None:
            eq = original_eq
            print((str(eq[16:32])+" "+str(eq[32:])).replace(".", "").replace("\n", ""))
            break  # Print only the first equation in the class
        else:
            continue
    print()

Representatives of equivalence classes

      Canonicalization has been applied,
      and thus we do no know the actual sign of the normal vector
      when the original equation cannot be found.
      It may point inwards or outwards.
      
Class 1:
[ 0  0  1  0  1 -1  0  0  0  0  0  0  0  0  0  1] [0 1 0 0]

Class 2:
[ 1  0  0  0  0 -1  1  0  0  0  0  0  0  0  0  1] [0 0 0 1]

Class 3:
[ 0  0  0  0  0  0  1  0  0  0  1  0 -1  1  1 -1] [0 0 1 0]

Class 4:
[ 0  0  0  0  0  0  1  0  1  0  0  0  0 -1  0  1] [0 0 0 1]

Class 5:
[ 0  0  1  0  1  0  0 -1  0  0  0  0  0  1  0  0] [ 0  0  0 -1]

Class 6:
[ 0  0  0  0  0  1  0  0  1  0  0  0  0  0  1 -1] [ 0 -1  0  0]

Class 7:
[ 1  0  0  0  0  0  0  0  0  0  0  0  0  1  1 -1] [ 0 -1  0  0]

Class 8:
[ 0  0  1  0  0  1  0 -1  0  0  0  0  1  0  0  0] [ 0  0  0 -1]

Class 9:
[ 1  0  0  0  1 -1 -1  1  0  0  0  0  1  0  0  0] [ 0  0 -1  0]

Class 10:
[ 1  0  0  0  1 -1  0  0  0  0  0  0  1  0 -1  1] [ 0  0 -1  0]

Class 11:
[ 0  0  1  0  0  0  0

In [33]:
256 == 2**8

True

### Observe more the equations to partition them

In [16]:
scaled_4 = scaled_equations[ np.count_nonzero(scaled_equations, axis=1) == 5 ]
scaled_6 = scaled_equations[ np.count_nonzero(scaled_equations, axis=1) == 7 ]

normalized_4 = normalized_equations[ np.count_nonzero(normalized_equations, axis=1) == 5 ]
normalized_6 = normalized_equations[ np.count_nonzero(normalized_equations, axis=1) == 7 ]

print(f"Scaled 4: {len(scaled_4)}")
print(f"Scaled 6: {len(scaled_6)}")
print(f"Normalized 4: {len(normalized_4)}")
print(f"Normalized 6: {len(normalized_6)}")

print(f"Sum along columns (scaled4):     {np.sum(scaled_4, axis=0)}")
print(f"Sum along columns (scaled6):     {np.sum(scaled_6, axis=0)}")
print(f"Sum along columns (normalized4): {np.sum(normalized_4, axis=0)}")
print(f"Sum along columns (normalized6): {np.sum(normalized_6, axis=0)}")

print(f"Sum along rows (scaled4):     {np.sum(scaled_4, axis=1)}")
print(f"Sum along rows (scaled6):     {np.sum(scaled_6, axis=1)}")
print(f"Sum along rows (normalized4): {np.sum(normalized_4, axis=1)}")
print(f"Sum along rows (normalized6): {np.sum(normalized_6, axis=1)}")

Scaled 4: 16
Scaled 6: 16
Normalized 4: 16
Normalized 6: 16
Sum along columns (scaled4):     [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 0 0 0 0]
Sum along columns (scaled6):     [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 0 0 0 0]
Sum along columns (normalized4): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 0 0 0 0]
Sum along columns (normalized6): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 4 0 0 0 0 0]
Sum along rows (scaled4):     [3 3 3 1 1 1 1 3 3 1 1 3 1 1 3 3]
Sum along rows (scaled6):     [3 1 1 3 3 3 3 3 1 1 1 1 1 1 3 3]
Sum along rows (normalized4): [3 3 3 1 1 1 1 3 3 1 1 3 1 1 3 3]
Sum along rows (normalized6): [3 1 1 3 3 3 3 3 1 1 1 1 1 1 3 3]


In [17]:
cor_sign_scaled_4 = scaled_4 * np.sum(scaled_4[:,-4:], axis=1).reshape(-1, 1)
cor_sign_scaled_6 = scaled_6 * np.sum(scaled_6[:,-4:], axis=1).reshape(-1, 1)
cor_sign_normal_4 = normalized_4 * np.sum(normalized_4[:,-4:], axis=1).reshape(-1, 1)
cor_sign_normal_6 = normalized_6 * np.sum(normalized_6[:,-4:], axis=1).reshape(-1, 1)


In [18]:
print(np.all(cor_sign_scaled_4 == cor_sign_normal_4))
print(np.all(cor_sign_scaled_6 == cor_sign_normal_6))


True
True


In [19]:
sorted_scaled = scaled_equations.tolist()
sorted_normal = normalized_equations.tolist()

sorted_scaled.sort(key=lambda x: tuple(x))
sorted_normal.sort(key=lambda x: tuple(x))

print("Sorted equations:")
for line in sorted_scaled:
    print(str(line[16:]).replace(",", "").replace("-1", "-").replace("1", "+").replace("0", " "))


print(sorted_scaled == sorted_normal)

Sorted equations:
[-   +     +         +       + - +      ]
[-   +     + + -     +           +      ]
[        - + +       +       + -     +  ]
[              +     +   + -       +    ]
[              + +         - +         +]
[            +       +   - + + -     +  ]
[            +   +         -   +       +]
[          +         +   +     -       -]
[          +     +           + -   -    ]
[          + + - +                 -    ]
[        + -   +     +             +    ]
[        +   - + +       + -         -  ]
[        +           +     +   -       -]
[        +       +       + - - +     -  ]
[    +   - + + -             +       +  ]
[    +           -   +     + + - +      ]
[    +                   + -   +   +    ]
[    +       + - -   +     +     +      ]
[    +       + -         - + +       +  ]
[    +     +   -         +             -]
[    +   + -                   +   +    ]
[    +   +     -           +           -]
[+   -         + +       + -     -      ]
[+   -   + -   +